In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import TensorDataset ,DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [8]:
transform = transforms.ToTensor()

trainset = datasets.MNIST(
    root="./Data",
    train=True,
    download=True,
    transform=transform
)

testset = datasets.MNIST(
    root="./Data",
    train=False,
    download=True,
    transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

In [38]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size = 128, num_layers = 2, num_classes = 10):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = out[:, -1, :]   # last time step
        out = self.fc(out)
        return out

In [39]:
input_size = 28
model = RNN(input_size)

In [40]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [43]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    for images, labels in trainloader:

        images = images.squeeze(1)  # for RNN input (batch, 28, 28)

        outputs = model(images)
        # outputs = torch.sigmoid(outputs.squeeze()) 

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(trainloader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")

Epoch [1/5], Training Loss: 0.4641
Epoch [2/5], Training Loss: 0.2217
Epoch [3/5], Training Loss: 0.2080
Epoch [4/5], Training Loss: 0.1712
Epoch [5/5], Training Loss: 0.1578


In [44]:
import torch

correct = 0
total = 0

model.eval()  # evaluation mode

with torch.no_grad():
    for images, labels in testloader:

        images = images.squeeze(1)  # for RNN input (28,28)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 96.55
